# 03 Establish Baselines


### 1. Goal And Setup

Question: How much of UdonPred's apparent performance is above trivial sequence signal,
random-label training, and annotation noise?


In [ ]:
# !pip install -r ../requirements.txt

from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results"


### 2. Load  or Calculate Raw UdonPred 7x7 Matrix

In [ ]:
# check if matrix already exists
matrix_csv = RESULTS / "udonpred_matrix" / "matrix.csv"
if not matrix_csv.exists():
    print("Running UdonPred matrix computation...")
    # !python ../scripts/run_udonpred_matrix.py --output_dir ../results/udonpred_matrix --device cpu
else:
    print("UdonPred matrix already exists, skipping computation.")

udon_matrix = pd.read_csv(RESULTS / "udonpred_matrix" / "matrix.csv")
udon_matrix

In [ ]:
metric_cols = [
    "trizod", "chezod", "softdis", "pdbflex",
    "atlas", "plddt", "disprot\n(AP)", "disprot\n(AUROC)"
]
plt.figure(figsize=(12, 5))
sns.heatmap(
    udon_matrix.set_index("train_dataset")[metric_cols],
    annot=True,
    fmt=".3f",
    cmap="viridis",
)
plt.title("Raw UdonPred 7x7 Performance")
plt.show()


### 3. Create Simple Baseline

Amino acid composition logistic regression estimates how much signal is recoverable from global sequence composition.
Coil propensity estimates how much local secondary-structure-like signal explains disorder labels.

The idea is to test the model in the same way as UdonPred to obtain a comparable 7x7 matrix.

In [ ]:
if (RESULTS / "simple_baselines" / "matrix.csv").exists():
    print("Baseline matrix already exists, skipping computation.")
else:
    print("Running simple baselines matrix computation...")
    # !python ../scripts/run_simple_baselines.py

baseline_matrix = pd.read_csv(RESULTS / "simple_baselines" / "matrix.csv")
baseline_matrix

In [ ]:
simple_plot = baseline_matrix.set_index(["baseline", "train_dataset"])[metric_cols]

plt.figure(figsize=(12, 5))
sns.heatmap(simple_plot, annot=True, fmt=".3f", cmap="viridis")
plt.xlabel("Test dataset")
plt.title("Simple Baseline Performance")
plt.show()


### 4. UdonPred Matrix As Headroom Above Baseline

The baseline models estimate how much performance can be explained by simple sequence signals, such as global amino-acid composition or local coil/hydrophobicity propensities. The headroom analysis asks how much performance remains after accounting for those simple signals.

For each test metric, we compute:

```text
headroom = UdonPred score - baseline score
```

Positive values mean that UdonPred adds predictive signal beyond the baseline. Values around zero mean that the simple baseline already explains most of the measured performance for that dataset/metric. Negative values mean that the selected baseline outperforms that UdonPred setting.

We use two views:

- `matched composition baseline`: compares each UdonPred training row to the amino-acid-composition baseline trained on the same dataset. This keeps the original 7x7 matrix structure.
- `best simple baseline`: compares UdonPred to the strongest available simple baseline for each test metric. This is the stricter and more conservative summary.


In [ ]:
# Convert both result tables to numeric matrices with matching columns.
udon_scores = udon_matrix.set_index("train_dataset")[metric_cols].apply(pd.to_numeric)
baseline_scores = baseline_matrix.set_index(["baseline", "train_dataset"])[metric_cols].apply(pd.to_numeric)

headroom_dir = RESULTS / "headroom_above_baseline"
headroom_dir.mkdir(parents=True, exist_ok=True)

print("UdonPred matrix shape:", udon_scores.shape)
print("Baseline matrix shape:", baseline_scores.shape)


#### 4.1 Headroom Against The Matched Composition Baseline

The amino-acid-composition baseline is trained once for each UdonPred training dataset. Therefore, it can be aligned row-by-row with the UdonPred matrix: `trizod` UdonPred is compared with `trizod_composition`, `chezod` UdonPred with `chezod_composition`, and so on.


In [ ]:
matched_rows = {}
missing_rows = []
for train_dataset in udon_scores.index:
    key = ("aa_composition_logreg", f"{train_dataset}_composition")
    if key in baseline_scores.index:
        matched_rows[train_dataset] = baseline_scores.loc[key]
    else:
        missing_rows.append(key)

matched_composition_baseline = pd.DataFrame.from_dict(matched_rows, orient="index").reindex(udon_scores.index)
headroom_vs_matched_composition = udon_scores - matched_composition_baseline

if missing_rows:
    print("Missing matched composition baseline rows:", missing_rows)

headroom_vs_matched_composition.to_csv(headroom_dir / "headroom_vs_matched_composition.csv")
display(headroom_vs_matched_composition.style.format("{:.3f}"))


In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(
    headroom_vs_matched_composition,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
)
plt.xlabel("Test dataset / metric")
plt.ylabel("UdonPred training dataset")
plt.title("UdonPred headroom above matched composition baseline")
plt.tight_layout()
plt.savefig(headroom_dir / "headroom_vs_matched_composition_heatmap.png", dpi=200)
plt.show()


#### 4.2 Headroom Against The Best Simple Baseline

The matched-composition view keeps the 7x7 structure, but it is not always the strongest baseline. For the final interpretation, we also compare UdonPred against the best simple baseline available for each test metric. This gives a stricter estimate of how much performance UdonPred adds above simple sequence-derived signal.


In [ ]:
best_baseline_scores = baseline_scores.max(axis=0)
best_baseline_rows = baseline_scores.idxmax(axis=0)

best_baseline_summary = pd.DataFrame({
    "best_baseline": [row[0] for row in best_baseline_rows],
    "best_baseline_train_dataset": [row[1] for row in best_baseline_rows],
    "best_baseline_score": best_baseline_scores,
})

best_baseline_summary.to_csv(headroom_dir / "best_baseline_per_metric.csv")
display(best_baseline_summary.style.format({"best_baseline_score": "{:.3f}"}))


In [ ]:
headroom_vs_best_baseline = udon_scores.subtract(best_baseline_scores, axis="columns")
headroom_vs_best_baseline.to_csv(headroom_dir / "headroom_vs_best_baseline.csv")

plt.figure(figsize=(12, 5))
sns.heatmap(
    headroom_vs_best_baseline,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
)
plt.xlabel("Test dataset / metric")
plt.ylabel("UdonPred training dataset")
plt.title("UdonPred headroom above best simple baseline")
plt.tight_layout()
plt.savefig(headroom_dir / "headroom_vs_best_baseline_heatmap.png", dpi=200)
plt.show()


#### 4.3 Compact Summary For The Report

The table below summarizes the most important comparison for each test metric: the best UdonPred score, the same-dataset UdonPred score, the best simple baseline score, and the remaining headroom.


In [ ]:
best_udon_scores = udon_scores.max(axis=0)
best_udon_train = udon_scores.idxmax(axis=0)

same_dataset_scores = pd.Series({
    metric: udon_scores.loc[metric.split("\n")[0], metric]
    if metric.split("\n")[0] in udon_scores.index else np.nan
    for metric in metric_cols
})

headroom_summary = pd.DataFrame({
    "best_udon_training_dataset": best_udon_train,
    "best_udon_score": best_udon_scores,
    "same_dataset_udon_score": same_dataset_scores,
    "best_baseline": best_baseline_summary["best_baseline"],
    "best_baseline_train_dataset": best_baseline_summary["best_baseline_train_dataset"],
    "best_baseline_score": best_baseline_scores,
})
headroom_summary["headroom_best_udon_vs_best_baseline"] = (
    headroom_summary["best_udon_score"] - headroom_summary["best_baseline_score"]
)
headroom_summary["headroom_same_dataset_vs_best_baseline"] = (
    headroom_summary["same_dataset_udon_score"] - headroom_summary["best_baseline_score"]
)

headroom_summary.to_csv(headroom_dir / "headroom_summary.csv")
display(headroom_summary.style.format({
    "best_udon_score": "{:.3f}",
    "same_dataset_udon_score": "{:.3f}",
    "best_baseline_score": "{:.3f}",
    "headroom_best_udon_vs_best_baseline": "{:.3f}",
    "headroom_same_dataset_vs_best_baseline": "{:.3f}",
}))


In [ ]:
summary_plot = headroom_summary.reset_index(names="test_metric")

plt.figure(figsize=(10, 4))
sns.barplot(
    data=summary_plot,
    x="test_metric",
    y="headroom_best_udon_vs_best_baseline",
    color="#4C78A8",
)
plt.axhline(0, color="black", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Test dataset / metric")
plt.ylabel("Best UdonPred - best baseline")
plt.title("Headroom above best simple baseline")
plt.tight_layout()
plt.savefig(headroom_dir / "headroom_summary_barplot.png", dpi=200)
plt.show()


In [ ]:
largest_headroom_metric = headroom_summary["headroom_best_udon_vs_best_baseline"].idxmax()
smallest_headroom_metric = headroom_summary["headroom_best_udon_vs_best_baseline"].idxmin()

print("Report notes:")
print(
    f"- Largest headroom: {largest_headroom_metric} "
    f"({headroom_summary.loc[largest_headroom_metric, 'headroom_best_udon_vs_best_baseline']:.3f})."
)
print(
    f"- Smallest headroom: {smallest_headroom_metric} "
    f"({headroom_summary.loc[smallest_headroom_metric, 'headroom_best_udon_vs_best_baseline']:.3f})."
)
print(
    f"- Mean headroom across metrics: "
    f"{headroom_summary['headroom_best_udon_vs_best_baseline'].mean():.3f}."
)


### 5. Interpretation Draft

This section evaluates UdonPred as headroom above simple baselines rather than only reporting raw prediction performance. The matched-composition comparison preserves the original cross-dataset matrix and asks whether each UdonPred model improves over a baseline trained from the same dataset. The best-baseline comparison is stricter: for each test metric, UdonPred is compared against whichever simple baseline performed best.

The main quantity is `best UdonPred score - best baseline score`. Positive headroom indicates that UdonPred learns information beyond simple amino-acid composition, coil propensity, hydrophobicity, or random scoring. Small or negative headroom suggests that the corresponding benchmark is strongly explainable by simple sequence statistics, or that the tested UdonPred model does not transfer well to that annotation type.

The files written to `results/headroom_above_baseline/` are the reusable outputs for this task: the two headroom matrices, the per-metric best-baseline table, and the compact summary table for the report.


### 6. Normalized Headroom Against Annotation Ceiling

The raw baseline headroom above is useful, but it does not account for how much agreement the annotations themselves allow. Here we normalize the best-simple-baseline delta by the metric-matched annotation ceiling:

```text
(UdonPred - best simple baseline) / (annotation ceiling - best simple baseline)
```

Diagonal cells use a ceiling of `1.0`. Off-diagonal dataset pairs without exact annotation overlap remain `NaN`; cells where the ceiling is not above the baseline are also left blank and described in `cell_status.csv`.


In [ ]:
import subprocess
import sys

normalized_dir = RESULTS / "normalized_headroom"
normalized_matrix_path = normalized_dir / "normalized_headroom_vs_best_simple_baseline.csv"
ceiling_summary_path = RESULTS / "annotation_ceiling" / "annotation_ceiling_summary.csv"

if not ceiling_summary_path.exists():
    print("Running annotation ceiling computation...")
    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "estimate_annotation_ceiling.py"),
        "--udonpred-dir", str(ROOT / "UdonPred"),
        "--output-dir", str(RESULTS / "annotation_ceiling"),
    ]
    subprocess.run(cmd, cwd=ROOT, check=True)

if not normalized_matrix_path.exists():
    print("Running normalized headroom computation...")
    cmd = [sys.executable, str(ROOT / "scripts" / "compute_normalized_headroom.py")]
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("Normalized headroom outputs already exist, loading existing files.")

normalized_headroom = pd.read_csv(normalized_matrix_path, index_col=0)
ceiling_matrix = pd.read_csv(normalized_dir / "ceiling_matrix.csv", index_col=0)
available_headroom = pd.read_csv(normalized_dir / "available_headroom_vs_best_simple_baseline.csv", index_col=0)
cell_status = pd.read_csv(normalized_dir / "cell_status.csv")
normalized_summary = pd.read_csv(normalized_dir / "normalized_headroom_summary.csv")

display(normalized_headroom.style.format("{:.3f}"))


In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(
    normalized_headroom,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    mask=normalized_headroom.isna(),
)
plt.xlabel("Test dataset / metric")
plt.ylabel("UdonPred training dataset")
plt.title("Normalized UdonPred headroom against annotation ceiling")
plt.tight_layout()
plt.savefig(normalized_dir / "normalized_headroom_vs_best_simple_baseline_heatmap.png", dpi=200)
plt.show()

print("Cell status counts:")
display(cell_status["status"].value_counts().to_frame("n"))
display(normalized_summary)


### 7. Shuffled-Label Null

The shuffled-label workflow estimates an empirical null for UdonPred heads trained on randomized residue labels. The current repository has one completed seed, so these numbers are exploratory and should not be treated as stable uncertainty intervals.


In [ ]:
shuffled_summary = pd.read_csv(normalized_dir / "shuffled_null_summary.csv")
shuffled_mean = pd.read_csv(normalized_dir / "shuffled_null_mean_matrix.csv", index_col=0)
udon_minus_shuffled = pd.read_csv(normalized_dir / "udon_minus_shuffled_null.csv", index_col=0)
normalized_vs_shuffled = pd.read_csv(normalized_dir / "normalized_headroom_vs_shuffled_null.csv", index_col=0)

if shuffled_summary["n"].max() == 1:
    print("Only one shuffled-label seed is available; interpret this null comparison as exploratory.")

display(shuffled_summary)
display(udon_minus_shuffled.style.format("{:.3f}"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(
    udon_minus_shuffled,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    ax=axes[0],
)
axes[0].set_title("Observed UdonPred minus shuffled-label null")
axes[0].set_xlabel("Test dataset / metric")
axes[0].set_ylabel("Training dataset")

sns.heatmap(
    normalized_vs_shuffled,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    mask=normalized_vs_shuffled.isna(),
    ax=axes[1],
)
axes[1].set_title("Normalized headroom using shuffled-label null")
axes[1].set_xlabel("Test dataset / metric")
axes[1].set_ylabel("Training dataset")

plt.tight_layout()
plt.savefig(normalized_dir / "shuffled_null_headroom_heatmaps.png", dpi=200)
plt.show()
